# ResNet50 Improved 7-Class Model

Refined version based on previous experiments. This notebook keeps all 7 HAM10000 classes and improves the existing ResNet setup without dropping classes.


In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam


2026-05-04 20:11:31.206965: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Cloud paths
PROJECT_ROOT = "/home/jovyan"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 7
EPOCHS = 15

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("Data files:", os.listdir(DATA_DIR))
print("Result files:", os.listdir(RESULTS_DIR))


DATA_DIR: /home/jovyan/data
RESULTS_DIR: /home/jovyan/results
MODELS_DIR: /home/jovyan/models
Data files: ['.ipynb_checkpoints', 'hmnist_28_28_L.csv', 'hmnist_28_28_RGB.csv', 'HAM10000_images_part_1', 'hmnist_8_8_RGB.csv', 'hmnist_8_8_L.csv', 'HAM10000_images_part_2', 'HAM10000_metadata.csv']
Result files: ['old splits', '.ipynb_checkpoints', 'train_split.csv', 'val_split.csv', 'test_split.csv']


## Load fixed split CSVs

These must be the original local split files uploaded to cloud:
- train: 7210
- validation: 802
- test: 2003


In [4]:
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "val_split.csv"))
test_df = pd.read_csv(os.path.join(RESULTS_DIR, "test_split.csv"))

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Train class distribution:")
print(train_df["dx"].value_counts())


Train: 7010
Validation: 1502
Test: 1503
Train class distribution:
dx
nv       4693
mel       779
bkl       769
bcc       360
akiec     229
vasc       99
df         81
Name: count, dtype: int64


In [5]:
# IMPORTANT: rebuild image paths for the cloud environment.
# The uploaded split CSVs may contain Mac paths in the old `path` column.
# We overwrite/create `image_path` using /home/jovyan/data.
IMG_DIR_1 = os.path.join(DATA_DIR, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(DATA_DIR, "HAM10000_images_part_2")

def get_image_path(image_id):
    path1 = os.path.join(IMG_DIR_1, image_id + ".jpg")
    path2 = os.path.join(IMG_DIR_2, image_id + ".jpg")
    if os.path.exists(path1):
        return path1
    if os.path.exists(path2):
        return path2
    return None

for df_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["image_path"] = df["image_id"].apply(get_image_path)
    missing = df["image_path"].isna().sum()
    print(f"{df_name} missing image paths:", missing)

print(train_df[["image_id", "dx", "label", "image_path"]].head())


train missing image paths: 0
val missing image paths: 0
test missing image paths: 0
       image_id   dx  label                                         image_path
0  ISIC_0031775   nv      5  /home/jovyan/data/HAM10000_images_part_2/ISIC_...
1  ISIC_0027306  mel      4  /home/jovyan/data/HAM10000_images_part_1/ISIC_...
2  ISIC_0033895   nv      5  /home/jovyan/data/HAM10000_images_part_2/ISIC_...
3  ISIC_0025491   nv      5  /home/jovyan/data/HAM10000_images_part_1/ISIC_...
4  ISIC_0031023  mel      4  /home/jovyan/data/HAM10000_images_part_2/ISIC_...


In [6]:
# Class mapping used in this project
label_map = {
    0: "akiec",
    1: "bcc",
    2: "bkl",
    3: "df",
    4: "mel",
    5: "nv",
    6: "vasc",
}
labels = [label_map[i] for i in range(NUM_CLASSES)]
labels


['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

## Class weights

We keep all seven classes, but cap class weights at 3.0 to reduce instability and avoid overcompensation toward very small classes.


In [7]:
classes = np.sort(train_df["label"].unique())
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

raw_class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

# Cap extreme class weights to reduce instability
class_weights = {
    int(cls): min(float(weight), 3.0)
    for cls, weight in zip(classes, class_weights_array)
}

print("Raw class weights:")
for k, v in raw_class_weights.items():
    print(f"{k} ({label_map[k]}): {v:.4f}")

print("Capped class weights:")
for k, v in class_weights.items():
    print(f"{k} ({label_map[k]}): {v:.4f}")


Raw class weights:
0 (akiec): 4.3731
1 (bcc): 2.7817
2 (bkl): 1.3022
3 (df): 12.3633
4 (mel): 1.2855
5 (nv): 0.2134
6 (vasc): 10.1154
Capped class weights:
0 (akiec): 3.0000
1 (bcc): 2.7817
2 (bkl): 1.3022
3 (df): 3.0000
4 (mel): 1.2855
5 (nv): 0.2134
6 (vasc): 3.0000


## TensorFlow datasets


In [8]:
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["label"].values)
)
val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["image_path"].values, val_df["label"].values)
)
test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["image_path"].values, test_df["label"].values)
)

train_ds = (
    train_ds
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

for images, y in train_ds.take(1):
    print("Image batch:", images.shape)
    print("Label batch:", y.shape)
    print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))


I0000 00:00:1777925505.212187    1740 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13294 MB memory:  -> device: 0, name: NVIDIA A16, pci bus id: 0000:06:00.0, compute capability: 8.6


Image batch: (32, 224, 224, 3)
Label batch: (32,)
Pixel range: 0.0 to 1.0


2026-05-04 20:11:48.654384: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Build improved ResNet50 model

Changes from previous version:
- keep all 7 classes
- cap class weights at 3.0
- unfreeze top 30 ResNet layers
- stronger but still controlled augmentation
- lower learning rate for stable fine-tuning
- early stopping + checkpointing


In [9]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")


In [10]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze top 30 layers only
for layer in base_model.layers[-30:]:
    layer.trainable = True

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)

# Improved classifier head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,147,591 (92.12 MB)

 Trainable params: 15,009,287 (57.26 MB)

 Non-trainable params: 9,138,304 (34.86 MB)

In [11]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "resnet50_7class_improved_best.keras"),
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

model.save(os.path.join(MODELS_DIR, "resnet50_7class_improved_final.keras"))
print("Saved final model.")


Epoch 1/15


2026-05-04 20:12:30.715212: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


220/220 ━━━━━━━━━━━━━━━━━━━━ 100s 346ms/step - accuracy: 0.2506 - loss: 1.7467 - val_accuracy: 0.0107 - val_loss: 2.2930 - learning_rate: 5.0000e-06
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 329ms/step - accuracy: 0.2857 - loss: 1.5514 - val_accuracy: 0.1431 - val_loss: 1.8990 - learning_rate: 5.0000e-06
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 331ms/step - accuracy: 0.3310 - loss: 1.5117 - val_accuracy: 0.1305 - val_loss: 2.4700 - learning_rate: 5.0000e-06
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 328ms/step - accuracy: 0.3184 - loss: 1.4815 - val_accuracy: 0.3462 - val_loss: 2.0670 - learning_rate: 5.0000e-06
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 77s 340ms/step - accuracy: 0.3295 - loss: 1.4118 - val_accuracy: 0.5113 - val_loss: 1.4741 - learning_rate: 2.5000e-06
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 330ms/step - accuracy: 0.3553 - loss: 1.3791 - val_accuracy: 0.4115 - val_loss: 2.0411 - learning_rate: 2.5000e-06
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 329ms/step -

## Evaluation

Use balanced accuracy, sensitivity/recall, specificity, and confusion matrix. Normal accuracy is shown but should not be the main metric due to class imbalance.


In [ ]:
best_model = tf.keras.models.load_model(
    os.path.join(MODELS_DIR, "resnet50_7class_improved_best.keras")
)

y_true = test_df["label"].values
y_prob = best_model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

accuracy = np.mean(y_true == y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)

print("Accuracy:", round(accuracy, 4))
print("Balanced accuracy:", round(bal_acc, 4))

print("
Classification report:")
print(classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("
Confusion matrix:")
display(cm_df)

cm_df.to_csv(os.path.join(RESULTS_DIR, "resnet50_7class_improved_confusion_matrix.csv"))


In [13]:
# Per-class sensitivity and specificity
metrics = []

for i, class_name in label_map.items():
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    metrics.append({
        "Class": class_name,
        "Precision": precision,
        "Sensitivity/Recall": sensitivity,
        "Specificity": specificity,
        "Support": np.sum(cm[i, :])
    })

metrics_df = pd.DataFrame(metrics)
display(metrics_df)
print("Balanced accuracy:", round(metrics_df["Sensitivity/Recall"].mean(), 4))

metrics_df.to_csv(os.path.join(RESULTS_DIR, "resnet50_7class_improved_metrics.csv"), index=False)


NameError: name 'label_map' is not defined

In [10]:
# Melanoma-focused result
mel_metrics = metrics_df[metrics_df["Class"] == "mel"]
display(mel_metrics)


NameError: name 'metrics_df' is not defined